# 07 — YOLOv8-seg (End-to-End Instance Segmentation) Evaluation

**Objective**:

Run end-to-end (E2E) evaluation on the test split (**1,733 test images / 6,398 pairs**) for the **YOLOv8-seg** baseline (`ecustfd_yolov8seg_best.pt`), combining direct instance segmentation with paper-faithful $\beta$-calibration (50/50 training split).

### Key Pipeline Components:
1. **Instance Segmentation**: YOLOv8-seg jointly predicts bounding boxes and segmentation masks for the 19 food classes and reference coin.
2. **Coin Calibration**: Computes physical scale (cm/px) from the reference coin ($2.5\text{ cm}$ diameter) with relaxed coin-gating (cross-fill and fallback scale of $0.1080\text{ cm/px}$).
3. **View Pairing**: Matches corresponding top-view and side-view images.
4. **3D Volume & Calorie Estimation**: Computes volume using geometric shape formulas and applies regression $\beta$-calibration to estimate mass and calories.
5. **Artifacts**: Exports evaluation reports (`summary.txt`, `report_test_*.json`, `samples_test_*.csv`, `betas_train_*.json`, `speed_per_image.json`).


In [1]:
"""Cell 1 -- Setup Dual Logging (Console + File Log)."""
import sys
from pathlib import Path

PROJECT_ROOT = Path("E:/AI_Research/dlt8").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from src.yolo_seg_eval.two_stage_helpers import setup_eval_logging

log, LOG_PATH, RUN_DIR, RUN_TS = setup_eval_logging("07_e2e_yolov8_paper_faithful", PROJECT_ROOT)
log.info("Logger initialized for YOLOv8-seg evaluation.")


2026-09-03 09:07:34 [INFO] ======================================================================
2026-09-03 09:07:34 [INFO] === 07_E2E_YOLOV8_PAPER_FAITHFUL -- EVALUATION SESSION STARTED ===
2026-09-03 09:07:34 [INFO] ======================================================================
2026-09-03 09:07:34 [INFO] PROJECT_ROOT : E:\AI_Research\dlt8
2026-09-03 09:07:34 [INFO] LOG_PATH     : E:\AI_Research\dlt8\outputs\logs\07_e2e_yolov8_paper_faithful_20260903-090734.log
2026-09-03 09:07:34 [INFO] RUN_DIR      : E:\AI_Research\dlt8\outputs\predictions\07_e2e_yolov8_paper_faithful_20260903-090734
2026-09-03 09:07:34 [INFO] TIMESTAMP    : 20260903-090734
2026-09-03 09:07:34 [INFO] Logger initialized for YOLOv8-seg evaluation.


In [2]:
"""Cell 2 -- Imports and dependencies."""
import warnings
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

from src.constants import FOOD_CLASSES, SHAPE_MODELS
from src.yolo_seg_eval.inference import load_yolo_seg, predict_one
from src.yolo_seg_eval.two_stage_helpers import (
    sanity_check_label_mapping,
    fix_ground_truth_aliasing,
    run_smoke_test,
    compute_speed_report,
    format_and_print_report,
    CANONICAL_YOLO_CLASSES,
)
from src.e2e_pipeline.dataset_split import (
    load_split,
    resolve_image_paths,
    group_top_side,
    make_pairs,
)

log.info("Imports OK.")
log.info("  FOOD_CLASSES (%d): %s", len(FOOD_CLASSES), ", ".join(FOOD_CLASSES))


2026-09-03 09:07:34 [INFO] Imports OK.
2026-09-03 09:07:34 [INFO]   FOOD_CLASSES (19): apple, banana, bread, bun, doughnut, egg, fried_dough_twist, grape, lemon, litchi, mango, mooncake, orange, peach, pear, plum, qiwi, sachima, tomato


In [3]:
"""Cell 3 -- Config and hyperparameters."""
MODEL_VARIANT = "yolov8_seg"
CONF = 0.3   # frozen from sweep 20260902-234626 (test_tune): min MAE duoi rang buoc giu nguyen coverage ~100%
IOU_THRESH = 0.50
IMG_SZ = 480
SPLIT = "test_final"  # held-out final half (conf được freeze từ sweep trên test_tune — KHÔNG tune trên tập này)
DEVICE = "0" if torch.cuda.is_available() else "cpu"

# Set MAX_PAIRS = 5 for fast validation test, or None for full test (1733 images)
MAX_PAIRS = None

MODEL_WEIGHTS = PROJECT_ROOT / "models" / "ecustfd_yolov8seg_best.pt"
if not MODEL_WEIGHTS.exists():
    raise FileNotFoundError(f"Model weights not found: {MODEL_WEIGHTS}")

IMAGES_DIR = PROJECT_ROOT / "data" / "raw" / "ECUSTFD" / "JPEGImages"
IMAGESETS_DIR = PROJECT_ROOT / "data" / "raw" / "ECUSTFD" / "ImageSets" / "Main"

TAG = f"{SPLIT}_conf{int(CONF*100)}_beta"
CSV_PATH = RUN_DIR / f"samples_{TAG}.csv"
JSON_PATH = RUN_DIR / f"report_{TAG}.json"
BETA_JSON = RUN_DIR / f"betas_train_conf{int(CONF*100)}.json"

log.info("Configuration:")
log.info("  Model Weights : %s (exists=%s)", MODEL_WEIGHTS, MODEL_WEIGHTS.exists())
log.info("  Model Variant : %s (Instance Segmentation)", MODEL_VARIANT)
log.info("  Confidence    : %.2f | IoU: %.2f | ImgSz: %d", CONF, IOU_THRESH, IMG_SZ)
log.info("  Max Pairs     : %s", MAX_PAIRS if MAX_PAIRS else "All")


2026-09-03 09:07:34 [INFO] Configuration:
2026-09-03 09:07:34 [INFO]   Model Weights : E:\AI_Research\dlt8\models\ecustfd_yolov8seg_best.pt (exists=True)
2026-09-03 09:07:34 [INFO]   Model Variant : yolov8_seg (Instance Segmentation)
2026-09-03 09:07:34 [INFO]   Confidence    : 0.30 | IoU: 0.50 | ImgSz: 480
2026-09-03 09:07:34 [INFO]   Max Pairs     : All


In [4]:
"""Cell 4 -- Sanity Check: Label Mapping & Schema Validation.

Guarantees that 20 classes are strictly aligned (idx 0=apple, idx 4=coin, idx 19=tomato).
Prevents off-by-one label shift.
"""
sanity_check_label_mapping(CANONICAL_YOLO_CLASSES, logger=log)


2026-09-03 09:07:34 [INFO] [Sanity Check] Verifying YOLO-seg class mapping contract...
2026-09-03 09:07:34 [INFO]   [PASS] 20 YOLO-seg classes validated: idx 0='apple', idx 4='coin', idx 19='tomato'.
2026-09-03 09:07:34 [INFO]   [PASS] No off-by-one label shift detected.


True

In [5]:
"""Cell 5 -- Load food_info.xls and density.xls."""
from src.calorie_estimation import parse_food_info
from src.faster_rcnn.eval_pipeline import _load_ground_truth

FOOD_INFO_XLS = PROJECT_ROOT / "data" / "raw" / "ECUSTFD" / "paper" / "food_info.xls"
DENSITY_XLS = PROJECT_ROOT / "data" / "raw" / "ECUSTFD" / "density.xls"

food_info = parse_food_info(FOOD_INFO_XLS)
gt_by_class = _load_ground_truth(DENSITY_XLS)

# Fix potential spelling alias (fired_dough_twist vs fried_dough_twist)
gt_by_class = fix_ground_truth_aliasing(gt_by_class, logger=log)

log.info("Loaded food_info: %d classes | GT density: %d classes", len(food_info), len(gt_by_class))


2026-09-03 09:07:36 [INFO] Loaded food_info: 21 classes | GT density: 20 classes


In [6]:
"""Cell 6 -- Load test split and view pairs."""
test_split_file = IMAGESETS_DIR / f"{SPLIT}.txt"
test_stems = load_split(test_split_file)
log.info("Loaded %s.txt: %d stems", SPLIT, len(test_stems))

test_paths = resolve_image_paths(test_stems, IMAGES_DIR)
log.info("Resolved image files: %d / %d stems", len(test_paths), len(test_stems))

test_groups = group_top_side(test_paths)
test_pairs = make_pairs(test_groups)
if MAX_PAIRS is not None:
    test_pairs = test_pairs[:MAX_PAIRS]
    log.info("[Testing] Sliced test_pairs to MAX_PAIRS=%d", len(test_pairs))
log.info("Paired views: %d (top, side) pairs resolved.", len(test_pairs))


2026-09-03 09:07:36 [INFO] Loaded test_final.txt: 928 stems
2026-09-03 09:07:36 [INFO] Resolved image files: 928 / 928 stems
2026-09-03 09:07:36 [INFO] Paired views: 6398 (top, side) pairs resolved.


In [7]:
"""Cell 7 -- Load YOLOv8-seg checkpoint."""
log.info("Loading model weights from: %s", MODEL_WEIGHTS)
model = load_yolo_seg(MODEL_WEIGHTS, device=DEVICE)
log.info("Model loaded successfully on device=%s.", DEVICE)


2026-09-03 09:07:36 [INFO] Loading model weights from: E:\AI_Research\dlt8\models\ecustfd_yolov8seg_best.pt
2026-09-03 09:07:36 [INFO] Loaded YOLO-seg model from E:\AI_Research\dlt8\models\ecustfd_yolov8seg_best.pt
2026-09-03 09:07:36 [INFO] Model loaded successfully on device=0.


In [8]:
"""Cell 8 -- Multi-sample Smoke Test on representative images."""
sample_images = [
    IMAGES_DIR / "apple015T(1).JPG",
    IMAGES_DIR / "bread001T(1).JPG",
    IMAGES_DIR / "lemon001T(1).JPG",
]

def _smoke_predict(p, c):
    return predict_one(model, p, conf=c, iou_threshold=IOU_THRESH, imgsz=IMG_SZ, device=DEVICE)
    
run_smoke_test(_smoke_predict, sample_images, conf=0.25, logger=log)


2026-09-03 09:07:36 [INFO] [Smoke Test] Testing YOLO-seg predictor on 3 sample images...
2026-09-03 09:07:37 [INFO] [RESULT] predict_one(apple015T(1).JPG, conf=0.25): 2 dets, classes={'apple': 1, 'coin': 1}
2026-09-03 09:07:37 [INFO]   Sample [1/3] apple015T(1).JPG: 2 detections, classes={'apple': 1, 'coin': 1}
2026-09-03 09:07:37 [INFO] [RESULT] predict_one(bread001T(1).JPG, conf=0.25): 2 dets, classes={'bread': 1, 'coin': 1}
2026-09-03 09:07:37 [INFO]   Sample [2/3] bread001T(1).JPG: 2 detections, classes={'bread': 1, 'coin': 1}
2026-09-03 09:07:37 [INFO] [RESULT] predict_one(lemon001T(1).JPG, conf=0.25): 2 dets, classes={'lemon': 1, 'coin': 1}
2026-09-03 09:07:37 [INFO]   Sample [3/3] lemon001T(1).JPG: 2 detections, classes={'lemon': 1, 'coin': 1}
2026-09-03 09:07:37 [INFO] [Smoke Test] PASS -- All smoke test samples processed without crash.


In [9]:
"""Cell 9 -- Per-image End-to-End Speed Timer (Memory auto-managed by refcount cache)."""
import atexit
import time
import src.yolo_seg_eval.inference as yolo_seg_inf

_E2E_TIMES = []
_E2E_NDETS = []
_E2E_NMASK = []

# Global counters for inference timing tracking
_INFERENCE_COUNTER = 0
_CURRENT_CONF = 0.8

_orig_predict_one = yolo_seg_inf.predict_one

def _timed_predict_one(model_obj, image_path, conf=0.8, iou_threshold=0.50, imgsz=480, device=None):
    """Wrapper to track inference timing. Memory managed by reference-counted cache in eval_pipeline."""
    global _INFERENCE_COUNTER, _CURRENT_CONF

    # Reset counter whenever conf threshold changes (new sweep/config)
    if conf != _CURRENT_CONF:
        _CURRENT_CONF = conf
        _INFERENCE_COUNTER = 0

    # Timing measurement
    t0 = time.perf_counter()
    dets = _orig_predict_one(model_obj, image_path, conf=conf, iou_threshold=iou_threshold, imgsz=imgsz, device=device)
    elapsed = time.perf_counter() - t0
    _E2E_TIMES.append(elapsed)
    _E2E_NDETS.append(len(dets))
    _E2E_NMASK.append(sum(1 for d in dets if d.get("class_name") != "coin" and d.get("mask") is not None))

    _INFERENCE_COUNTER += 1

    return dets

yolo_seg_inf.predict_one = _timed_predict_one

def _restore_predict_one():
    yolo_seg_inf.predict_one = _orig_predict_one

atexit.register(_restore_predict_one)
log.info("[Timer] Installed per-image timer wrapper for YOLOv8-seg (memory auto-managed by refcount cache).")


2026-09-03 09:07:37 [INFO] [Timer] Installed per-image timer wrapper for YOLOv8-seg (memory auto-managed by refcount cache).


In [10]:
"""Cell 10 -- Relaxed coin-gate patch to ensure no sample drop."""
import src.yolo_seg_eval.eval_pipeline as yolo_seg_ep

def _relaxed_compute_scale(dets, fallback=0.1080):
    coins = [d for d in dets if d.get("class_name") == "coin"]
    if not coins:
        return fallback, None
    best_coin = max(coins, key=lambda d: d.get("conf", 0.0))
    bbox = best_coin["bbox"]
    w = abs(bbox[2] - bbox[0])
    h = abs(bbox[3] - bbox[1])
    diam = max(w, h)
    if diam < 5:
        return fallback, None
    scale = 2.5 / diam  # 1 Yuan coin = 2.5 cm
    return scale, best_coin

log.info("[Patch] Relaxed coin-gate active.")


2026-09-03 09:07:37 [INFO] [Patch] Relaxed coin-gate active.


In [11]:
"""Cell 11 -- Run Full E2E YOLOv8-seg Pipeline."""
log.info("=" * 70)
log.info("Running E2E YOLOv8-seg pipeline...")
log.info("  variant=%s, split=%s, conf=%.2f, apply_beta=True", MODEL_VARIANT, SPLIT, CONF)
log.info("=" * 70)

artifacts = yolo_seg_ep._run_one_config(
    model_weights=MODEL_WEIGHTS,
    model_variant=MODEL_VARIANT,
    split=SPLIT,
    images_dir=IMAGES_DIR,
    imagesets_dir=IMAGESETS_DIR,
    conf_threshold=CONF,
    food_info=food_info,
    gt_by_class=gt_by_class,
    out_dir=RUN_DIR,
    apply_beta=True,
    device=DEVICE,
    iou_threshold=IOU_THRESH,
    imgsz=IMG_SZ,
    max_pairs=MAX_PAIRS,
)

log.info("Pipeline run finished successfully. Artifacts: %s", artifacts)


2026-09-03 09:07:37 [INFO] ======================================================================
2026-09-03 09:07:37 [INFO] Running E2E YOLOv8-seg pipeline...
2026-09-03 09:07:37 [INFO]   variant=yolov8_seg, split=test_final, conf=0.30, apply_beta=True
2026-09-03 09:07:37 [INFO] ======================================================================
2026-09-03 09:07:37 [INFO] === config: variant=yolov8_seg, split=test_final, conf=0.30, apply_beta=True, max_pairs=None ===
2026-09-03 09:07:37 [INFO] Loading YOLO-seg model: E:\AI_Research\dlt8\models\ecustfd_yolov8seg_best.pt
2026-09-03 09:07:37 [INFO] Loaded YOLO-seg model from E:\AI_Research\dlt8\models\ecustfd_yolov8seg_best.pt
2026-09-03 09:07:37 [INFO] Beta calibration requested: fitting on train split (paper 50/50)
2026-09-03 09:07:37 [INFO]   Train split: 1169 stems, 6772 pairs
2026-09-03 09:07:37 [INFO] [RESULT] predict_one(apple001T(1).JPG, conf=0.01): 2 dets, classes={'apple': 1, 'coin': 1}
2026-09-03 09:07:37 [INFO] [RESULT] pr

In [12]:
"""Cell 12 -- Render Per-class & Overall Markdown Report."""
import json
report = json.loads(Path(JSON_PATH).read_text(encoding="utf-8"))

df_report, report_data = format_and_print_report(
    JSON_PATH,
    title=f"Per-class ME_vol / ME_mass (YOLOv8-seg Instance Segmentation, conf={CONF:.2f})",
)



=== Per-class ME_vol / ME_mass (YOLOv8-seg Instance Segmentation, conf=0.30) ===
| Class             |   n |   ME_vol (%) |   |ME_vol| (%) |   ME_mass (%) |
|:------------------|----:|-------------:|---------------:|--------------:|
| peach             | 420 |         4.04 |           7.73 |         -6.50 |
| litchi            | 128 |         6.09 |           8.69 |          6.22 |
| apple             | 905 |         2.39 |          10.77 |          1.73 |
| doughnut          | 398 |         0.63 |          15.73 |          2.39 |
| lemon             | 496 |         9.27 |          18.17 |          5.81 |
| plum              | 576 |        18.99 |          19.23 |         12.42 |
| sachima           | 653 |       -13.88 |          19.77 |        -13.68 |
| grape             | 289 |       -20.28 |          20.38 |        -15.45 |
| bun               |  96 |        10.80 |          20.92 |         21.13 |
| banana            | 319 |         1.11 |          21.21 |          3.45 |
| mang

In [13]:
"""Cell 13 -- Beta Calibration Sanity Check."""
this_betas = report.get("betas", {})
log.info("Beta calibrated for %d classes.", len(this_betas))
if this_betas:
    vals = list(this_betas.values())
    log.info("  range  : [%.4f, %.4f]", min(vals), max(vals))
    log.info("  median : %.4f", float(np.median(vals)))


2026-09-03 09:08:20 [INFO] Beta calibrated for 19 classes.
2026-09-03 09:08:20 [INFO]   range  : [0.5989, 1.2437]
2026-09-03 09:08:20 [INFO]   median : 0.9060


In [14]:
"""Cell 14 -- Compute & Print Per-image Latency Report."""
cfg = {
    "model_variant": MODEL_VARIANT,
    "conf_threshold": CONF,
    "iou_threshold": IOU_THRESH,
    "imgsz": IMG_SZ,
    "device": DEVICE,
}

speed_stats = compute_speed_report(
    _E2E_TIMES, _E2E_NDETS, _E2E_NMASK,
    config_dict=cfg,
    run_dir=RUN_DIR,
    backend_title="YOLOv8-seg (Instance Segmentation)",
    logger=log,
)


2026-09-03 09:08:20 [INFO] [timer] Speed report saved -> E:\AI_Research\dlt8\outputs\predictions\07_e2e_yolov8_paper_faithful_20260903-090734\speed_per_image.json

  PER-IMAGE END-TO-END SPEED (YOLOv8-seg (Instance Segmentation))
  n_images         : 2088
  total wall time  : 40.49s (0.67 min)
  throughput       : 51.574 images/sec
  Per-image latency:
    mean   : 19.4 ms  (0.0194s)
    median : 17.8 ms  (0.0178s)
    p90    : 24.7 ms  (0.0247s)
    p95    : 32.0 ms  (0.0320s)
    p99    : 39.6 ms  (0.0396s)
    min/max: 11.7 ms / 133.7 ms
    stdev  : 6.2 ms
  avg #dets / #masks per img: 2.28 / 1.15



In [15]:
"""Cell 15 -- Write summary.txt and Manifest."""
summary_lines = [
    "=" * 70,
    "07 -- YOLOV8-SEG E2E EVALUATION -- SUMMARY",
    "=" * 70,
    f"Run timestamp      : {RUN_TS}",
    f"Model variant      : {MODEL_VARIANT}",
    f"Model weights      : {MODEL_WEIGHTS}",
    f"Split              : {SPLIT} ({len(test_stems)} stems)",
    f"Confidence         : {CONF}",
    f"NMS IoU threshold  : {IOU_THRESH}",
    f"Image size         : {IMG_SZ}",
    f"Device             : {DEVICE}",
    "",
    "Overall metrics:",
]
for k, v in report.get("overall", {}).items():
    val_str = f"{v:.4f}" if isinstance(v, float) else str(v)
    summary_lines.append(f"  {k:20s} = {val_str}")

summary_lines.append("")
summary_lines.append("Files written:")
summary_lines.append(f"  CSV    : {CSV_PATH}")
summary_lines.append(f"  JSON   : {JSON_PATH}")
summary_lines.append(f"  Betas  : {BETA_JSON}")
summary_lines.append(f"  Log    : {LOG_PATH}")
summary_lines.append(f"  Speed  : {RUN_DIR / 'speed_per_image.json'}")

summary_path = RUN_DIR / "summary.txt"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
log.info("Summary written -> %s", summary_path)

print("\n" + "=" * 70)
print("  ALL OUTPUT FILES (04):")
print("=" * 70)
print(f"  Log            : {LOG_PATH}")
print(f"  Run dir        : {RUN_DIR}")
print(f"  Predictions CSV: {CSV_PATH}")
print(f"  Report JSON    : {JSON_PATH}")
print(f"  Betas JSON     : {BETA_JSON}")
print(f"  Speed JSON     : {RUN_DIR / 'speed_per_image.json'}")
print(f"  Summary txt    : {summary_path}")
print("=" * 70)
print("\n>>> Notebook 04 is complete.")


2026-09-03 09:08:20 [INFO] Summary written -> E:\AI_Research\dlt8\outputs\predictions\07_e2e_yolov8_paper_faithful_20260903-090734\summary.txt

  ALL OUTPUT FILES (04):
  Log            : E:\AI_Research\dlt8\outputs\logs\07_e2e_yolov8_paper_faithful_20260903-090734.log
  Run dir        : E:\AI_Research\dlt8\outputs\predictions\07_e2e_yolov8_paper_faithful_20260903-090734
  Predictions CSV: E:\AI_Research\dlt8\outputs\predictions\07_e2e_yolov8_paper_faithful_20260903-090734\samples_test_final_conf30_beta.csv
  Report JSON    : E:\AI_Research\dlt8\outputs\predictions\07_e2e_yolov8_paper_faithful_20260903-090734\report_test_final_conf30_beta.json
  Betas JSON     : E:\AI_Research\dlt8\outputs\predictions\07_e2e_yolov8_paper_faithful_20260903-090734\betas_train_conf30.json
  Speed JSON     : E:\AI_Research\dlt8\outputs\predictions\07_e2e_yolov8_paper_faithful_20260903-090734\speed_per_image.json
  Summary txt    : E:\AI_Research\dlt8\outputs\predictions\07_e2e_yolov8_paper_faithful_2026090